
## Folder 07 — Gold Business Analytics:


This notebook creates a **business-ready daily sales fact table** from your **Silver orders** and **order items**.

It calculates:

- Daily order count.
- Daily sales value.
- Average order value (AOV).
- Delivered, shipped, processing and cancelled order counts.
- Sales by channel.
- Month-to-date sales.
- Running cumulative sales.


**Important business definition: **

- This notebook calculates sales using silver_orders.order_total. 
- It represents order value, not actual cash collected. 
- Payment refunds, failed transactions, taxes and shipping adjustments are not automatically reconciled here.

In [0]:
## 1. Import libraries

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

print("Libraries imported successfully.")


## 2. Project configuration

CATALOG = "workspace"

SILVER_SCHEMA = "ecommerce_silver"
GOLD_SCHEMA = "ecommerce_gold"

ORDERS_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.silver_orders"
)

ORDER_ITEMS_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.silver_order_items"
)

GOLD_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.gold_daily_sales"
)

RUN_ID = spark.sql(
    "SELECT uuid() AS run_id"
).first()["run_id"]

RUN_TIMESTAMP = datetime.now()

print("Run ID:", RUN_ID)


## 3. Validate source tables

if not spark.catalog.tableExists(ORDERS_TABLE):
    raise Exception(f"Missing table: {ORDERS_TABLE}")

if not spark.catalog.tableExists(ORDER_ITEMS_TABLE):
    raise Exception(f"Missing table: {ORDER_ITEMS_TABLE}")

# Create Gold schema if it does not exist.

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}"
)

print("Source tables validated.")
print("Gold schema ready.")


## 4. Load Silver datasets

orders_df = spark.table(ORDERS_TABLE)

order_items_df = spark.table(ORDER_ITEMS_TABLE)

print("Orders:", orders_df.count())
print("Order items:", order_items_df.count())


## 5. Validate required columns

required_order_columns = [
    "order_id",
    "order_timestamp",
    "order_status",
    "order_total",
    "sales_channel",
    "currency"
]

required_item_columns = [
    "order_id",
    "order_item_id",
    "quantity",
    "line_total"
]

for column_name in required_order_columns:

    if column_name not in orders_df.columns:
        raise Exception(
            f"Missing orders column: {column_name}"
        )

for column_name in required_item_columns:

    if column_name not in order_items_df.columns:
        raise Exception(
            f"Missing order_items column: {column_name}"
        )


## 6. Standardize order data

orders_clean_df = (
    orders_df
    .withColumn(
        "order_date",
        F.to_date("order_timestamp")
    )
    .withColumn(
        "order_status",
        F.upper(F.trim(F.col("order_status")))
    )
    .withColumn(
        "sales_channel",
        F.coalesce(
            F.initcap(F.trim(F.col("sales_channel"))),
            F.lit("Unknown")
        )
    )
    .withColumn(
        "currency",
        F.upper(F.trim(F.col("currency")))
    )
    .withColumn(
        "order_total",
        F.col("order_total").cast("decimal(18,2)")
    )
    .filter(
        F.col("order_id").isNotNull()
        & F.col("order_date").isNotNull()
        & F.col("order_total").isNotNull()
        & (F.col("order_total") >= 0)
        & F.col("currency").isNotNull()
    )
)


## 7. Deduplicate orders

#Silver should already contain unique order IDs.

# This is a defensive check before Gold aggregation.

order_window = (
    Window
    .partitionBy("order_id")
    .orderBy(
        F.col("_ingestion_timestamp").desc_nulls_last()
        if "_ingestion_timestamp" in orders_clean_df.columns
        else F.col("order_timestamp").desc()
    )
)

orders_deduplicated_df = (
    orders_clean_df
    .withColumn(
        "_row_number",
        F.row_number().over(order_window)
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)


## 8. Prepare order item data

order_items_clean_df = (
    order_items_df
    .withColumn(
        "quantity",
        F.col("quantity").cast("int")
    )
    .withColumn(
        "line_total",
        F.col("line_total").cast("decimal(18,2)")
    )
    .filter(
        F.col("order_id").isNotNull()
        & F.col("order_item_id").isNotNull()
        & F.col("quantity").isNotNull()
        & (F.col("quantity") > 0)
        & F.col("line_total").isNotNull()
        & (F.col("line_total") >= 0)
    )
)


## 9. Aggregate order item metrics

# Aggregate child records before joining them to orders.
# This prevents one-to-many joins from multiplying order totals.

order_item_metrics_df = (
    order_items_clean_df
    .groupBy("order_id")
    .agg(
        F.countDistinct("order_item_id").alias(
            "order_item_count"
        ),

        F.sum("quantity").alias(
            "total_quantity"
        ),

        F.sum("line_total").cast("decimal(18,2)").alias(
            "item_sales_value"
        )
    )
)


## 10. Join orders with item metrics

orders_with_items_df = (
    orders_deduplicated_df.alias("orders")
    .join(
        order_item_metrics_df.alias("items"),
        F.col("orders.order_id")
        == F.col("items.order_id"),
        "left"
    )
    .select(
        F.col("orders.order_id"),
        F.col("orders.order_date"),
        F.col("orders.order_status"),
        F.col("orders.sales_channel"),
        F.col("orders.currency"),
        F.col("orders.order_total"),
        F.coalesce(
            F.col("items.order_item_count"),
            F.lit(0)
        ).alias("order_item_count"),
        F.coalesce(
            F.col("items.total_quantity"),
            F.lit(0)
        ).alias("total_quantity"),
        F.col("items.item_sales_value")
    )
)


## 11. Create daily sales aggregation

# Cancelled orders are excluded from sales value.

# Their counts remain available as a separate operational metric.


daily_sales_df = (
    orders_with_items_df
    .groupBy(
        "order_date",
        "currency"
    )
    .agg(

        F.countDistinct("order_id").alias(
            "total_orders"
        ),

        F.sum(
            F.when(
                F.col("order_status") != "CANCELLED",
                F.col("order_total")
            ).otherwise(F.lit(0))
        ).cast("decimal(18,2)").alias(
            "gross_order_value"
        ),

        F.sum(
            F.when(
                F.col("order_status") != "CANCELLED",
                1
            ).otherwise(0)
        ).alias(
            "non_cancelled_orders"
        ),

        F.sum(
            F.when(
                F.col("order_status") == "CANCELLED",
                1
            ).otherwise(0)
        ).alias(
            "cancelled_orders"
        ),

        F.sum(
            F.when(
                F.col("order_status") == "DELIVERED",
                1
            ).otherwise(0)
        ).alias(
            "delivered_orders"
        ),

        F.sum(
            F.when(
                F.col("order_status") == "SHIPPED",
                1
            ).otherwise(0)
        ).alias(
            "shipped_orders"
        ),

        F.sum(
            F.when(
                F.col("order_status") == "PROCESSING",
                1
            ).otherwise(0)
        ).alias(
            "processing_orders"
        ),

        F.sum(
            F.when(
                F.col("order_status") != "CANCELLED",
                F.col("total_quantity")
            ).otherwise(0)
        ).alias(
            "total_units_sold"
        ),

        F.sum(
            F.when(
                F.col("order_status") != "CANCELLED",
                F.col("order_item_count")
            ).otherwise(0)
        ).alias(
            "total_order_items"
        )
    )
)


## 12. Calculate average order value

daily_sales_df = (
    daily_sales_df
    .withColumn(
        "average_order_value",
        F.when(
            F.col("non_cancelled_orders") > 0,
            F.round(
                F.col("gross_order_value")
                / F.col("non_cancelled_orders"),
                2
            )
        ).otherwise(F.lit(0))
    )
)


## 13. Calculate sales by channel

channel_sales_df = (
    orders_with_items_df
    .groupBy(
        "order_date",
        "currency",
        "sales_channel"
    )
    .agg(

        F.countDistinct("order_id").alias(
            "channel_total_orders"
        ),

        F.sum(
            F.when(
                F.col("order_status") != "CANCELLED",
                F.col("order_total")
            ).otherwise(F.lit(0))
        ).cast("decimal(18,2)").alias(
            "channel_sales_value"
        )
    )
)


## 14. Calculate monthly and cumulative sales

monthly_window = (
    Window
    .partitionBy(
        F.year("order_date"),
        F.month("order_date"),
        "currency"
    )
    .orderBy("order_date")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

daily_sales_df = (
    daily_sales_df
    .withColumn(
        "month_to_date_sales",
        F.sum("gross_order_value").over(
            monthly_window
        )
    )
)

cumulative_window = (
    Window
    .partitionBy("currency")
    .orderBy("order_date")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

daily_sales_df = (
    daily_sales_df
    .withColumn(
        "cumulative_sales",
        F.sum("gross_order_value").over(
            cumulative_window
        )
    )
)


## 15. Add audit columns

daily_sales_df = (
    daily_sales_df
    .withColumn(
        "run_id",
        F.lit(RUN_ID)
    )
    .withColumn(
        "created_at",
        F.lit(RUN_TIMESTAMP)
    )
)

channel_sales_df = (
    channel_sales_df
    .withColumn(
        "run_id",
        F.lit(RUN_ID)
    )
    .withColumn(
        "created_at",
        F.lit(RUN_TIMESTAMP)
    )
)


## 16. Display daily sales

display(
    daily_sales_df.orderBy(
        F.col("order_date").desc(),
        "currency"
    )
)


## 17. Display channel sales

display(
    channel_sales_df.orderBy(
        F.col("order_date").desc(),
        "channel_sales_value"
    )
)


## 18. Save Gold daily sales table

(
    daily_sales_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TABLE)
)

print("Gold daily sales table created:", GOLD_TABLE)


## 19. Save channel sales table

CHANNEL_GOLD_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.gold_channel_daily_sales"
)

(
    channel_sales_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(CHANNEL_GOLD_TABLE)
)

print("Gold channel sales table created:", CHANNEL_GOLD_TABLE)


## 20. Final validation

gold_df = spark.table(GOLD_TABLE)

channel_gold_df = spark.table(CHANNEL_GOLD_TABLE)

print("Daily sales records:", gold_df.count())

print("Channel sales records:", channel_gold_df.count())

print("Total daily sales value:")

display(
    gold_df
    .groupBy("currency")
    .agg(
        F.sum("gross_order_value").alias(
            "total_order_value"
        ),
        F.sum("total_orders").alias(
            "total_orders"
        ),
        F.sum("cancelled_orders").alias(
            "cancelled_orders"
        )
    )
)

print("=" * 65)
print("GOLD DAILY SALES COMPLETED")
print("=" * 65)

print("Daily sales table:", GOLD_TABLE)
print("Channel sales table:", CHANNEL_GOLD_TABLE)



### Databricks SQL dashboard queries


**Query 1 — Daily sales trend**

In [0]:
%sql
SELECT
    order_date,
    currency,
    gross_order_value,
    total_orders,
    average_order_value
FROM workspace.ecommerce_gold.gold_daily_sales
ORDER BY order_date;


**Query 2 — Sales by channel**

In [0]:
%sql
SELECT
    sales_channel,
    SUM(channel_sales_value) AS total_sales,
    SUM(channel_total_orders) AS total_orders
FROM workspace.ecommerce_gold.gold_channel_daily_sales
GROUP BY sales_channel
ORDER BY total_sales DESC;


**Query 3 — Order status analysis**

In [0]:
%sql

SELECT
    currency,
    SUM(gross_order_value) AS total_sales,
    SUM(total_orders) AS total_orders,
    SUM(non_cancelled_orders) AS non_cancelled_orders,
    SUM(cancelled_orders) AS cancelled_orders,
    SUM(total_units_sold) AS total_units_sold
FROM workspace.ecommerce_gold.gold_daily_sales
GROUP BY currency;


**Query 4 — Monthly sales**

In [0]:
%sql
SELECT
    YEAR(order_date) AS sales_year,
    MONTH(order_date) AS sales_month,
    currency,
    SUM(gross_order_value) AS monthly_sales
FROM workspace.ecommerce_gold.gold_daily_sales
GROUP BY
    YEAR(order_date),
    MONTH(order_date),
    currency
ORDER BY sales_year, sales_month;


### 1. Execution Results



| Metric | Result |
|---|---:|
| Silver Orders Loaded | 19,800 |
| Silver Order Items Loaded | 19,793 |
| Gold Daily Sales Records | 730 |
| Gold Channel Sales Records | 3,636 |
| Total Sales Value | ₹96,76,75,435.56 |
| Cancelled Orders | 3,289 |
| Runtime | 20.066 seconds |


### 2. Execution Summary


```text
┌──────────────────────────────────────────────┐
│             GOLD LAYER EXECUTION             │
│                 SUCCESSFUL                   │
├──────────────────────────────────────────────┤
│  Silver Orders Loaded       : 19,800         │
│  Silver Order Items Loaded  : 19,793         │
│  Gold Daily Sales Records   : 730            │
│  Gold Channel Sales Records: 3,636           │
│                                              │
│  Total Sales Value          : ₹96.76 Crore   │
│  Cancelled Orders           : 3,289          │
│  Runtime                    : 20.066 Sec     │
└──────────────────────────────────────────────┘
```

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_gold.gold_daily_sales;